# Step 06 — Four search algorithms on London road network


In [1]:
import sys
import pickle
from pathlib import Path

import networkx as nx
import pandas as pd
from IPython.display import display

import routing
from routing.config import ORIGIN, DESTINATION, RESULTS_DIR, MIN_PROBE_EDGE_LENGTH_M, MAP_RADIUS_M
from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_pipeline import make_routing_graph
from routing.algorithms import run_all_algorithms
from routing.route_view import add_route_overlay

print("Python:", sys.version)
print("Kernel:", sys.executable)
print("Routing package:", Path(routing.__file__).resolve())


Python: 3.14.8 (main, Sep 30 2026, 17:55:09) [Clang 21.0.0 (clang-2100.3.34.2)]
Kernel: /Users/mojtabashabani/wiser/mary/.venv/bin/python
Routing package: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/src/routing/__init__.py


## 1 — Load OSM and the saved traffic snapshot first


In [2]:
RADIUS_M = MAP_RADIUS_M

G = load_road_graph(radius_m=RADIUS_M)

snapshot_path = RESULTS_DIR / "traffic_snapshot.pkl"
if not snapshot_path.exists():
    raise FileNotFoundError(
        f"{snapshot_path} was not found. Run Notebook 05 with RUN_LIVE_TRAFFIC=True first."
    )

with snapshot_path.open("rb") as file:
    traffic_bundle = pickle.load(file)

if isinstance(traffic_bundle, dict):
    snapshot = traffic_bundle.get("snapshot")
    summary = traffic_bundle.get("summary", {}) or {}
    plan = traffic_bundle.get("plan", [])
    saved_at = traffic_bundle.get("saved_at_utc", "timestamp not recorded")
elif isinstance(traffic_bundle, nx.MultiDiGraph):
    # Backward compatibility with older projects that saved the graph directly.
    snapshot = traffic_bundle
    summary = {}
    plan = []
    saved_at = "legacy graph file; timestamp not recorded"
else:
    raise TypeError(
        "Unsupported traffic snapshot format: "
        f"{type(traffic_bundle).__name__}"
    )

if not isinstance(snapshot, nx.MultiDiGraph):
    raise TypeError("Traffic bundle does not contain a valid MultiDiGraph snapshot")

assert isinstance(G, nx.MultiDiGraph)
if set(G.nodes) != set(snapshot.nodes):
    raise ValueError("Saved snapshot nodes do not match the configured OSM graph.")
if set(G.edges(keys=True)) != set(snapshot.edges(keys=True)):
    raise ValueError("Saved snapshot directed edges do not match the configured OSM graph.")

print("OSM graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "directed edges")
print("Snapshot:", snapshot.number_of_nodes(), "nodes,", snapshot.number_of_edges(), "directed edges")
print("Saved usable observations:", summary.get(
    "cumulative_usable_observed_edges",
    summary.get("usable_observed_edges", "not reported")
))
print("Saved at:", saved_at)
print("Snapshot file modified:", pd.Timestamp(snapshot_path.stat().st_mtime, unit="s", tz="UTC"))
print("PASS: The saved traffic snapshot matches the configured OSM graph")

OSM graph: 2569 nodes, 5850 directed edges
Snapshot: 2569 nodes, 5850 directed edges
Saved usable observations: 23
Saved at: 2026-11-09T10:51:34.525525+00:00
Snapshot file modified: 2026-11-09 10:51:34.541944027+00:00
PASS: The saved traffic snapshot matches the configured OSM graph


## 2 — Inspect the traffic observations before routing


In [3]:
from collections import Counter

status_counts = Counter()
observed_rows = []

for u, v, key, attrs in snapshot.edges(keys=True, data=True):
    status = attrs.get("traffic_status", "missing")
    status_counts[status] += 1

    usable = (
        attrs.get("traffic_matched") is True
        and attrs.get("time_is_imputed") is False
        and status in {"red", "yellow", "clear", "closed"}
    )

    if usable:
        observed_rows.append({
            "edge_id": (u, v, key),
            "status": status,
            "match_source": attrs.get("traffic_match_source"),
            "current_speed_kph": attrs.get("current_speed_kph"),
            "free_flow_speed_kph": attrs.get("free_flow_speed_kph"),
            "travel_time_s": attrs.get("travel_time_s"),
            "is_closed": attrs.get("is_closed", False),
        })

print("Traffic status counts:", dict(status_counts))
print("Usable observed directed edges:", len(observed_rows))
print("Total directed edges:", snapshot.number_of_edges())

observed_edges_df = pd.DataFrame(observed_rows)
display(observed_edges_df)


Traffic status counts: {'missing': 5827, 'clear': 13, 'yellow': 10}
Usable observed directed edges: 23
Total directed edges: 5850


,edge_id,status,match_source,current_speed_kph,free_flow_speed_kph,travel_time_s,is_closed
0,"(109869, 26389013, 0)",clear,direct_probe,27.0,27.0,11.756003,False
1,"(276550, 860183709, 0)",clear,direct_probe,19.0,19.0,19.057131,False
2,"(20947178, 1691189798, 0)",yellow,reverse_inferred,18.0,26.0,17.870665,False
3,"(25291778, 1360237602, 0)",yellow,direct_probe,13.0,18.0,18.664727,False
4,"(26389375, 34519894, 0)",clear,reverse_inferred,23.0,23.0,28.391868,False
5,"(34519894, 6280192964, 0)",clear,reverse_inferred,23.0,23.0,41.764058,False
6,"(265565434, 109870, 0)",clear,direct_probe,24.0,24.0,10.837494,False
7,"(265565438, 265565434, 0)",clear,direct_probe,24.0,24.0,15.566292,False
8,"(267342172, 265565438, 0)",clear,direct_probe,24.0,24.0,6.076877,False
9,"(385312258, 12195919, 0)",yellow,reverse_inferred,19.0,31.0,10.857511,False


## 3 — Choose the origin/destination on the map


In [4]:
route_widget, route_state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    traffic_graph=snapshot,
    max_snap_distance_m=350.0,
)

display(route_widget)


In [5]:
source = route_state["origin_node"]
goal = route_state["destination_node"]

print("Origin node:", source)
print("Destination node:", goal)
print("Last double-click:", route_state["last_click_latlon"])

assert source in G
assert goal in G

print("PASS: Valid origin and destination nodes")


Origin node: 25291778
Destination node: 26389375
Last double-click: None
PASS: Valid origin and destination nodes


## 4 — Build all routing graphs from the loaded traffic snapshot


In [6]:
measured_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="measured",
)

measured_distance = make_routing_graph(
    snapshot,
    objective="distance",
    scenario="measured",
)

free_flow_time = make_routing_graph(
    snapshot,
    objective="time",
    scenario="free_flow",
)

print("Measured-time edges:", measured_time.number_of_edges())
print("Measured-distance edges:", measured_distance.number_of_edges())
print("Free-flow edges:", free_flow_time.number_of_edges())

print("PASS: Routing graphs were built from the saved traffic snapshot")


Measured-time edges: 5850
Measured-distance edges: 5850
Free-flow edges: 5850
PASS: Routing graphs were built from the saved traffic snapshot


## 5 — Run BFS, DFS, Greedy Best-First Search, and A*


In [7]:
results_time = run_all_algorithms(measured_time, source, goal)
results_distance = run_all_algorithms(measured_distance, source, goal)
results_free = run_all_algorithms(free_flow_time, source, goal)

result_columns = [
    "algorithm",
    "found",
    "hops",
    "distance_m",
    "travel_time_s",
    "objective_cost",
    "imputed_route_edges",
    "expanded_nodes",
    "max_frontier_size",
    "runtime_ms",
]

def results_to_frame(results, scenario_name):
    rows = []
    for name, result in results.items():
        row = {key: result.get(key) for key in result_columns}
        row["scenario"] = scenario_name
        rows.append(row)
    return pd.DataFrame(rows)

comparison_df = pd.concat(
    [
        results_to_frame(results_time, "Measured Time"),
        results_to_frame(results_distance, "Measured Distance"),
        results_to_frame(results_free, "Free Flow"),
    ],
    ignore_index=True,
)

display(comparison_df.round(3))

if results_time["A*"]["found"]:
    print("A* time-optimal path:", results_time["A*"]["path_edges"])
else:
    print("No directed path exists for the selected origin/destination.")

print("PASS: All four algorithms were executed on all three routing graphs")


,algorithm,found,hops,distance_m,travel_time_s,objective_cost,imputed_route_edges,expanded_nodes,max_frontier_size,runtime_ms,scenario
0,bfs,True,5,679.322,88.963,88.963,4,44,18,0.167,Measured Time
1,DFS,True,209,15561.308,1892.831,1892.831,206,610,162,1.780,Measured Time
2,Greedy,True,5,679.322,88.963,88.963,4,6,8,3.985,Measured Time
3,A*,True,5,679.322,88.963,88.963,4,15,8,3.601,Measured Time
4,bfs,True,5,679.322,88.963,679.322,4,44,18,0.130,Measured Distance
5,DFS,True,209,15561.308,1892.831,15561.308,206,610,162,1.999,Measured Distance
6,Greedy,True,5,679.322,88.963,679.322,4,6,8,3.956,Measured Distance
7,A*,True,6,628.181,93.411,628.181,4,9,10,3.549,Measured Distance
8,bfs,True,5,679.322,82.437,82.437,4,44,18,0.130,Free Flow
9,DFS,True,209,15561.308,1877.467,1877.467,206,610,162,1.834,Free Flow


A* time-optimal path: [(25291778, 5898995362, 0), (5898995362, 2360746811, 0), (2360746811, 6280192964, 0), (6280192964, 34519894, 0), (34519894, 26389375, 0)]
PASS: All four algorithms were executed on all three routing graphs


## 6 — Draw the time-objective routes on the map


In [8]:
for name, result in results_time.items():
    if result["found"]:
        add_route_overlay(
            route_widget,
            G,
            result,
            name=f"{name} | measured-time weights",
        )
        print(name, "route drawn; edges:", len(result["path_edges"]))
    else:
        print(name, "did not find a route")

print("Routes were added to the interactive map.")


BFS route drawn; edges: 5
DFS route drawn; edges: 209
Greedy route drawn; edges: 5
A* route drawn; edges: 5
Routes were added to the interactive map.


## 7 — Simulate one directed road closure


In [9]:
best_route = results_time["A*"]

if best_route["found"] and best_route["path_edges"]:
    chosen_closure = best_route["path_edges"][len(best_route["path_edges"]) // 2]

    closure_graph = make_routing_graph(
        snapshot,
        objective="time",
        scenario="closure",
        imposed_closure=chosen_closure,
    )

    assert not closure_graph.has_edge(*chosen_closure)

    closure_results = run_all_algorithms(closure_graph, source, goal)

    print("Simulated directed closure:", chosen_closure)
    display(pd.DataFrame([
        {
            "algorithm": name,
            "found": result["found"],
            "hops": result["hops"],
            "travel_time_s": result["travel_time_s"],
            "distance_m": result["distance_m"],
        }
        for name, result in closure_results.items()
    ]).round(3))
else:
    print("Closure simulation skipped because A* did not find a nonempty route.")


Simulated directed closure: (2360746811, 6280192964, 0)


,algorithm,found,hops,travel_time_s,distance_m
0,BFS,True,6,93.411,628.181
1,DFS,True,384,3499.106,29008.973
2,Greedy,True,6,93.411,628.181
3,A*,True,6,93.411,628.181


## 8 — Validate route structures


In [10]:
for graph, results in (
    (measured_time, results_time),
    (measured_distance, results_distance),
    (free_flow_time, results_free),
):
    for name, result in results.items():
        if not result["found"]:
            continue

        assert result["path_nodes"][0] == source
        assert result["path_nodes"][-1] == goal
        assert len(result["path_edges"]) == len(result["path_nodes"]) - 1
        assert all(graph.has_edge(*edge) for edge in result["path_edges"])
        assert all(G.has_edge(*edge) for edge in result["path_edges"])

print("PASS: All found routes have valid directed edge sequences")


PASS: All found routes have valid directed edge sequences


## 9 — Build an Observed-Only graph




In [11]:
def build_observed_only_graph(source_graph):
    graph = source_graph.copy()
    allowed_sources = {"direct_probe", "reverse_inferred"}
    usable_statuses = {"red", "yellow", "clear"}

    remove_edges = []

    for u, v, key, attrs in graph.edges(keys=True, data=True):
        valid_observation = (
            attrs.get("traffic_matched") is True
            and attrs.get("time_is_imputed") is False
            and attrs.get("traffic_status") in usable_statuses
            and attrs.get("traffic_match_source") in allowed_sources
            and attrs.get("is_closed") is not True
        )

        if not valid_observation:
            remove_edges.append((u, v, key))

    graph.remove_edges_from(remove_edges)
    return graph


observed_only = build_observed_only_graph(snapshot)

print("========== OBSERVED-ONLY GRAPH ==========")
print("Nodes:", observed_only.number_of_nodes())
print("Edges:", observed_only.number_of_edges())

observed_route_exists = (
    source in observed_only
    and goal in observed_only
    and nx.has_path(observed_only, source, goal)
)

print("Observed-only route exists:", observed_route_exists)

if not observed_route_exists:
    print(
        "Expected with sparse coverage: observed edges exist, "
        "but they do not yet form a continuous directed route."
    )


========== OBSERVED-ONLY GRAPH ==========
Nodes: 2569
Edges: 23
Observed-only route exists: False
Expected with sparse coverage: observed edges exist, but they do not yet form a continuous directed route.


## 10 — Route Audit: inspect the exact edges used by each algorithm


In [12]:
route_audit_rows = []

scenario_results = {
    "Measured Time": results_time,
    "Measured Distance": results_distance,
    "Free Flow": results_free,
}

for scenario_name, results in scenario_results.items():
    for algorithm, result in results.items():
        if not result["found"]:
            continue

        for u, v, key in result["path_edges"]:
            attrs = snapshot[u][v][key]

            usable_observation = (
                attrs.get("traffic_matched") is True
                and attrs.get("time_is_imputed") is False
                and attrs.get("traffic_status") in {"red", "yellow", "clear"}
                and attrs.get("traffic_match_source") in {
                    "direct_probe",
                    "reverse_inferred",
                }
            )

            route_audit_rows.append({
                "scenario": scenario_name,
                "algorithm": algorithm,
                "edge_id": (u, v, key),
                "traffic_status": attrs.get("traffic_status", "missing"),
                "match_source": attrs.get("traffic_match_source"),
                "match_reason": attrs.get("traffic_match_reason"),
                "current_speed_kph": attrs.get("current_speed_kph"),
                "free_flow_speed_kph": attrs.get("free_flow_speed_kph"),
                "travel_time_s": attrs.get("travel_time_s"),
                "imputed": attrs.get("time_is_imputed", True),
                "usable_observation": usable_observation,
            })

route_audit_df = pd.DataFrame(route_audit_rows)

print("Route-audit rows:", len(route_audit_df))
display(route_audit_df)


Route-audit rows: 673


,scenario,algorithm,edge_id,traffic_status,match_source,match_reason,current_speed_kph,free_flow_speed_kph,travel_time_s,imputed,usable_observation
0,Measured Time,BFS,"(25291778, 5898995362, 0)",missing,NaN,probe_point_too_far,NaN,NaN,13.720352,True,False
1,Measured Time,BFS,"(5898995362, 2360746811, 0)",missing,NaN,short_segment,NaN,NaN,1.153502,True,False
2,Measured Time,BFS,"(2360746811, 6280192964, 0)",yellow,direct_probe,candidate_match,19.0,28.0,20.303092,False,True
3,Measured Time,BFS,"(6280192964, 34519894, 0)",missing,NaN,heading_proxy_mismatch,NaN,NaN,32.019111,True,False
4,Measured Time,BFS,"(34519894, 26389375, 0)",missing,NaN,heading_proxy_mismatch,NaN,NaN,21.767099,True,False
...,...,...,...,...,...,...,...,...,...,...,...
668,Free Flow,A*,"(25291778, 5898995362, 0)",missing,NaN,probe_point_too_far,NaN,NaN,13.720352,True,False
669,Free Flow,A*,"(5898995362, 2360746811, 0)",missing,NaN,short_segment,NaN,NaN,1.153502,True,False
670,Free Flow,A*,"(2360746811, 6280192964, 0)",yellow,direct_probe,candidate_match,19.0,28.0,20.303092,False,True
671,Free Flow,A*,"(6280192964, 34519894, 0)",missing,NaN,heading_proxy_mismatch,NaN,NaN,32.019111,True,False


In [13]:
if not route_audit_df.empty:
    route_coverage_df = (
        route_audit_df
        .groupby(["scenario", "algorithm"], as_index=False)
        .agg(
            route_edges=("edge_id", "count"),
            observed_edges=("usable_observation", "sum"),
            imputed_edges=("imputed", "sum"),
        )
    )

    route_coverage_df["observed_coverage_percent"] = (
        100 * route_coverage_df["observed_edges"]
        / route_coverage_df["route_edges"]
    ).round(2)

    print("========== ROUTE TRAFFIC COVERAGE ==========")
    display(route_coverage_df)
else:
    print("No route edges were available for audit.")


========== ROUTE TRAFFIC COVERAGE ==========


,scenario,algorithm,route_edges,observed_edges,imputed_edges,observed_coverage_percent
0,Free Flow,A*,5,1,4,20.00
1,Free Flow,BFS,5,1,4,20.00
2,Free Flow,DFS,209,3,206,1.44
3,Free Flow,Greedy,5,1,4,20.00
4,Measured Distance,A*,6,2,4,33.33
5,Measured Distance,BFS,5,1,4,20.00
6,Measured Distance,DFS,209,3,206,1.44
7,Measured Distance,Greedy,5,1,4,20.00
8,Measured Time,A*,5,1,4,20.00
9,Measured Time,BFS,5,1,4,20.00


In [14]:
# ==========================================
# ROUTE EDGE AUDIT - NO API REQUESTS
# ==========================================

print("========== FULL A* ROUTE EDGE AUDIT ==========")

route_edges = results_time["A*"]["path_edges"]

audit_rows = []

for raw_edge in route_edges:

    u, v, k = tuple(raw_edge)

    attrs = snapshot[u][v][k]

    usable = (
        attrs.get("traffic_matched") is True
        and attrs.get("time_is_imputed") is False
        and attrs.get("traffic_status")
            in {"red", "yellow", "clear", "closed"}
    )

    highway = attrs.get("highway")
    length = float(attrs.get("length", 0))

    reverse_data = snapshot.get_edge_data(v, u, default={})

    reverse_observations = []

    for rk, reverse_attrs in reverse_data.items():

        reverse_usable = (
            reverse_attrs.get("traffic_matched") is True
            and reverse_attrs.get("time_is_imputed") is False
            and reverse_attrs.get("traffic_status")
                in {"red", "yellow", "clear", "closed"}
        )

        if reverse_usable:
            reverse_observations.append({
                "reverse_edge": (v, u, rk),
                "status": reverse_attrs.get("traffic_status"),
                "speed": reverse_attrs.get("current_speed_kph"),
                "source": reverse_attrs.get("traffic_match_source"),
            })

    if usable:
        reason = "already_observed"
    elif length < MIN_PROBE_EDGE_LENGTH_M:
        reason = "too_short"
    else:
        reason = "needs_probe"

    audit_rows.append({
        "edge_id": (u, v, k),
        "length_m": round(length, 2),
        "highway": highway,
        "traffic_status": attrs.get("traffic_status"),
        "traffic_matched": attrs.get("traffic_matched"),
        "match_source": attrs.get("traffic_match_source"),
        "time_is_imputed": attrs.get("time_is_imputed"),
        "usable": usable,
        "reverse_observations": reverse_observations,
        "decision": reason,
    })

display(pd.DataFrame(audit_rows).to_string(index=False))

print("\nRoute edge count:", len(route_edges))
print(
    "Already observed:",
    sum(x["decision"] == "already_observed" for x in audit_rows)
)
print(
    "Too short:",
    sum(x["decision"] == "too_short" for x in audit_rows)
)
print(
    "Needs probe:",
    sum(x["decision"] == "needs_probe" for x in audit_rows)
)


========== FULL A* ROUTE EDGE AUDIT ==========


"                    edge_id  length_m                       highway traffic_status  traffic_matched match_source  time_is_imputed  usable                                                                                          reverse_observations         decision\n  (25291778, 5898995362, 0)    114.34                  unclassified        missing            False          NaN             True   False                                                                                                            []      needs_probe\n(5898995362, 2360746811, 0)      9.61                    trunk_link        missing            False          NaN             True   False                                                                                                            []        too_short\n(2360746811, 6280192964, 0)    107.16                         trunk         yellow             True direct_probe            False    True                                                                


Route edge count: 5
Already observed: 1
Too short: 1
Needs probe: 3


## 11 — Prepare route-focused probes for the next collection round



In [15]:
from routing.map_matching import directed_osm_line
import json

route_probe_rows = []
rejected_route_edges = []

SUPPORTED_HIGHWAY_CLASSES = {
    "motorway", "motorway_link", "trunk", "trunk_link",
    "primary", "primary_link", "secondary", "secondary_link",
    "tertiary", "tertiary_link", "unclassified", "residential",
    "living_street",
}

def _has_supported_highway(value):
    values = value if isinstance(value, (list, tuple, set)) else [value]
    return any(str(item).strip().lower() in SUPPORTED_HIGHWAY_CLASSES for item in values if item is not None)

astar_result = results_time["A*"]

if astar_result["found"]:
    seen_route_edges = set()
    for raw_edge_id in astar_result["path_edges"]:
        edge_id = tuple(raw_edge_id)
        if edge_id in seen_route_edges:
            continue
        seen_route_edges.add(edge_id)
        u, v, key = edge_id
        attrs = snapshot[u][v][key]

        usable_observation = (
            attrs.get("traffic_matched") is True
            and attrs.get("time_is_imputed") is False
            and attrs.get("traffic_status") in {"red", "yellow", "clear", "closed"}
            and attrs.get("traffic_match_source") in {"direct_probe", "reverse_inferred"}
        )
        if usable_observation:
            continue

        edge_length = float(attrs.get("length", 0.0))
        highway = attrs.get("highway")
        if edge_length < MIN_PROBE_EDGE_LENGTH_M:
            rejected_route_edges.append({
                "edge_id": list(edge_id),
                "length_m": round(edge_length, 2),
                "reason": "below_min_probe_edge_length",
            })
            continue
        if not _has_supported_highway(highway):
            rejected_route_edges.append({
                "edge_id": list(edge_id),
                "length_m": round(edge_length, 2),
                "reason": "unsupported_highway_class",
            })
            continue

        line = directed_osm_line(G, u, v, key)
        midpoint = line.interpolate(0.5, normalized=True)
        route_probe_rows.append({
            "edge_id": edge_id,
            "latitude": round(midpoint.y, 7),
            "longitude": round(midpoint.x, 7),
            "length_m": round(edge_length, 2),
            "road_class": highway,
            "current_traffic_status": attrs.get("traffic_status", "missing"),
            "time_is_imputed": attrs.get("time_is_imputed", True),
            "probe_fraction": 0.5,
        })

route_probe_candidates_df = pd.DataFrame(route_probe_rows)
route_probe_edge_ids = [tuple(row["edge_id"]) for row in route_probe_rows]

print("A* route edges needing traffic observations:", len(route_probe_edge_ids))
print("Rejected short/unsupported route edges:", len(rejected_route_edges))
if rejected_route_edges:
    display(pd.DataFrame(rejected_route_edges))
display(route_probe_candidates_df)

route_probe_path = RESULTS_DIR / "route_probe_candidates.json"
route_probe_payload = [
    {**row, "edge_id": list(row["edge_id"])}
    for row in route_probe_rows
]
route_probe_path.parent.mkdir(parents=True, exist_ok=True)
route_probe_path.write_text(
    json.dumps(route_probe_payload, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Route-focused probe candidates exported:", route_probe_path)


A* route edges needing traffic observations: 3
Rejected short/unsupported route edges: 1


,edge_id,length_m,reason
0,"[5898995362, 2360746811, 0]",9.61,below_min_probe_edge_length


,edge_id,latitude,longitude,length_m,road_class,current_traffic_status,time_is_imputed,probe_fraction
0,"(25291778, 5898995362, 0)",51.495059,-0.174919,114.34,unclassified,missing,True,0.5
1,"(6280192964, 34519894, 0)",51.496979,-0.173807,266.83,"[unclassified, living_street]",missing,True,0.5
2,"(34519894, 26389375, 0)",51.498980,-0.174206,181.39,unclassified,missing,True,0.5


Route-focused probe candidates exported: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/route_probe_candidates.json


In [16]:
print("========== NOTEBOOK 06 EXPORT CHECK ==========")

print("Origin:", source)
print("Destination:", goal)

print(
    "A* route edges:",
    len(results_time["A*"]["path_edges"])
)

print(
    "Exported candidate count:",
    len(route_probe_payload)
)

print("Export path:", route_probe_path.resolve())

========== NOTEBOOK 06 EXPORT CHECK ==========
Origin: 25291778
Destination: 26389375
A* route edges: 5
Exported candidate count: 3
Export path: /Users/mojtabashabani/wiser/AiWithPython/AiLeason/data/results/route_probe_candidates.json
